# Factory Waste Reduction | Silver Cleaning & Validation

## Purpose
This notebook standardizes data types and labels, combines 2025–2026 operational data, and validates the analysis-ready Silver layer. Gold reporting is intentionally kept in a separate notebook.

## 1. Create clean daily-consumption table

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_daily_consumption AS
SELECT TO_DATE(date) AS usage_date, TRIM(facility) AS facility, TRIM(machine_line) AS machine_line,
       INITCAP(TRIM(shift)) AS shift, TRIM(sku) AS sku,
       CAST(units_issued AS INT) AS units_issued, CAST(units_used AS INT) AS units_used,
       CAST(units_returned_to_stock AS INT) AS units_returned_to_stock, CAST(units_processed AS INT) AS units_processed,
       2025 AS source_year
FROM bronze_daily_consumption_2025
UNION ALL
SELECT TO_DATE(date), TRIM(facility), TRIM(machine_line), INITCAP(TRIM(shift)), TRIM(sku),
       CAST(units_issued AS INT), CAST(units_used AS INT), CAST(units_returned_to_stock AS INT),
       CAST(units_processed AS INT), 2026
FROM bronze_daily_consumption_2026;

## 2. Create clean waste-log table

The original waste-reason value is retained for auditability while the standardized value is used in reporting.

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_waste_log AS
SELECT TO_DATE(date) AS usage_date, TRIM(facility) AS facility, TRIM(machine_line) AS machine_line,
       INITCAP(TRIM(shift)) AS shift, TRIM(sku) AS sku, CAST(units_wasted AS INT) AS units_wasted,
       waste_reason AS waste_reason_original,
       CASE
         WHEN LOWER(TRIM(waste_reason)) IN ('over dispensing','over-dispensing') THEN 'Over-dispensing'
         WHEN LOWER(TRIM(waste_reason)) = 'label misprint' THEN 'Label Misprint'
         WHEN LOWER(TRIM(waste_reason)) = 'strap breakage' THEN 'Strap Breakage'
         WHEN LOWER(TRIM(waste_reason)) = 'tape misfeed' THEN 'Tape Misfeed'
         ELSE INITCAP(TRIM(waste_reason))
       END AS waste_reason,
       TRIM(reported_by) AS reported_by, 2025 AS source_year
FROM bronze_waste_log_2025
UNION ALL
SELECT TO_DATE(date), TRIM(facility), TRIM(machine_line), INITCAP(TRIM(shift)), TRIM(sku), CAST(units_wasted AS INT),
       waste_reason,
       CASE WHEN LOWER(TRIM(waste_reason)) IN ('over dispensing','over-dispensing') THEN 'Over-dispensing'
            WHEN LOWER(TRIM(waste_reason)) = 'label misprint' THEN 'Label Misprint'
            WHEN LOWER(TRIM(waste_reason)) = 'strap breakage' THEN 'Strap Breakage'
            WHEN LOWER(TRIM(waste_reason)) = 'tape misfeed' THEN 'Tape Misfeed'
            ELSE INITCAP(TRIM(waste_reason)) END,
       TRIM(reported_by), 2026
FROM bronze_waste_log_2026;

## 3. Create clean product-master and purchase-order tables

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_product_master AS
SELECT TRIM(sku) AS sku, TRIM(manufacturer) AS manufacturer, TRIM(product_name) AS product_name,
       TRIM(vendor) AS vendor, TRIM(category) AS category, TRIM(unit_of_measure) AS unit_of_measure,
       TRIM(product_spec) AS product_spec, CAST(case_pack AS INT) AS case_pack,
       CAST(standard_unit_cost AS DECIMAL(12,2)) AS standard_unit_cost, TRIM(pricing_basis) AS pricing_basis,
       CAST(minimum_order_units AS INT) AS minimum_order_units,
       CAST(standard_lead_time_days AS INT) AS standard_lead_time_days,
       CAST(reorder_point_units AS INT) AS reorder_point_units, CAST(machine_line_count AS INT) AS machine_line_count,
       CAST(operating_shifts_per_day AS INT) AS operating_shifts_per_day,
       CAST(units_issued_per_line_shift AS INT) AS units_issued_per_line_shift, TRIM(eligible_lines) AS eligible_lines
FROM manufacturing_cost_reduction.product_master;

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_purchase_orders AS
SELECT TRIM(po_id) AS po_id, TO_DATE(order_date) AS order_date, TO_DATE(received_date) AS received_date,
       TRIM(facility) AS facility, TRIM(sku) AS sku, CAST(qty_ordered AS INT) AS qty_ordered,
       CAST(unit_price AS DECIMAL(12,2)) AS unit_price, CAST(freight_cost AS DECIMAL(12,2)) AS freight_cost,
       CAST(qty_ordered * unit_price AS DECIMAL(14,2)) AS material_order_cost,
       CAST((qty_ordered * unit_price) + freight_cost AS DECIMAL(14,2)) AS total_order_cost,
       DATEDIFF(TO_DATE(received_date), TO_DATE(order_date)) AS lead_time_days,
       TRIM(order_type) AS order_type, TRIM(vendor) AS vendor, 2025 AS source_year
FROM bronze_purchase_orders_2025
UNION ALL
SELECT TRIM(po_id), TO_DATE(order_date), TO_DATE(received_date), TRIM(facility), TRIM(sku),
       CAST(qty_ordered AS INT), CAST(unit_price AS DECIMAL(12,2)), CAST(freight_cost AS DECIMAL(12,2)),
       CAST(qty_ordered * unit_price AS DECIMAL(14,2)), CAST((qty_ordered * unit_price) + freight_cost AS DECIMAL(14,2)),
       DATEDIFF(TO_DATE(received_date), TO_DATE(order_date)), TRIM(order_type), TRIM(vendor), 2026
FROM bronze_purchase_orders_2026;

## 4. Silver validation checks

These checks should return zero problem rows before the Gold analysis is refreshed.

In [ ]:
%sql
SELECT 'invalid_waste_rows' AS check_name, COUNT(*) AS problem_rows
FROM silver_waste_log WHERE usage_date IS NULL OR sku IS NULL OR units_wasted <= 0
UNION ALL
SELECT 'invalid_consumption_rows', COUNT(*)
FROM silver_daily_consumption WHERE usage_date IS NULL OR sku IS NULL OR units_issued < 0 OR units_used < 0
UNION ALL
SELECT 'unmatched_waste_skus', COUNT(*)
FROM silver_waste_log w LEFT ANTI JOIN silver_product_master p ON w.sku = p.sku
UNION ALL
SELECT 'duplicate_product_skus', COUNT(*)
FROM (SELECT sku FROM silver_product_master GROUP BY sku HAVING COUNT(*) > 1);

In [ ]:
%sql
SELECT source_year, waste_reason, COUNT(*) AS waste_events, SUM(units_wasted) AS units_wasted
FROM silver_waste_log
GROUP BY source_year, waste_reason
ORDER BY source_year, units_wasted DESC;

## Silver handoff

The next notebook joins clean consumption, waste, and product-cost data to answer business questions and build the Tableau data source.